In [2]:
# !pip install -U sentence-transformers

# Sentence Similarity Assessment

I need to assess the similarity between the generated explanations to estimate the final model uncertainty. These are some examples of possible explanations for the following code line ```max_with_square_ending_here = max(max_with_square_ending_here + nums[i], max_with_square_continue, max_with_square_start)```:
- The line updates the variable `max_with_square_ending_here` by taking the maximum of three values:
    1. The sum of `max_with_square_ending_here` and the current element `nums[i]`. This represents extending the existing square subsequence ending at the previous index by including the current element.
    2. `max_with_square_continue`, which likely holds the value of the maximum square subsequence that can continue without starting anew.
    3. `max_with_square_start`, which likely holds the value of the maximum square subsequence that starts with the current element.
    
    This approach ensures that `max_with_square_ending_here` always contains the maximum possible value for a square subsequence ending at the current index, considering all possible ways to extend or start a new subsequence.
    
- This line updates the variable `max_with_square_ending_here` to be the maximum of three possible values:
    1. The sum of the current value of `max_with_square_ending_here` and the current element `nums[i]`. This represents extending the existing square subarray with the current element.
    2. The value of `max_with_square_continue`, which could represent continuing a previous square subarray in some context.
    3. The value of `max_with_square_start`, which could represent starting a new square subarray with the current element.
    
    The purpose of this line is to determine the maximum value that can be achieved by either extending the current square subarray, continuing a previous one, or starting a new one with the current element. This is a common pattern in dynamic programming problems where subarray or substring properties are being optimized.
    
- This line updates the variable `max_with_square_ending_here` by taking the maximum of three possible values:
    1. The sum of the current value of `max_with_square_ending_here` and the current element `nums[i]`. This option represents extending the existing sequence that ends with a square number.
    2. The value of `max_with_square_continue`, which presumably holds the maximum sum of a sequence that continues from the previous elements without breaking the square number condition.
    3. The value of `max_with_square_start`, which represents starting a new sequence with the current element `nums[i]`.
    
    The purpose of this line is to determine the maximum sum of a sequence ending at the current element `nums[i]` while ensuring that the sequence can be extended or started anew if necessary, all the while maintaining the condition that the sequence includes square numbers.

In [12]:
import re
import pandas as pd
from sentence_transformers import SentenceTransformer

In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [58]:
explanations = [
    'The line updates the variable `max_with_square_ending_here` by taking the maximum of three values:\n1. The sum of `max_with_square_ending_here` and the current element `nums[i]`. This represents extending the existing square subsequence ending at the previous index by including the current element.\n2. `max_with_square_continue`, which likely holds the value of the maximum square subsequence that can continue without starting anew.\n3. `max_with_square_start`, which likely holds the value of the maximum square subsequence that starts with the current element.\n\nThis approach ensures that `max_with_square_ending_here` always contains the maximum possible value for a square subsequence ending at the current index, considering all possible ways to extend or start a new subsequence.',
    'This line updates the variable `max_with_square_ending_here` to be the maximum of three possible values:\n1. The sum of the current value of `max_with_square_ending_here` and the current element `nums[i]`. This represents extending the existing square subarray with the current element.\n2. The value of `max_with_square_continue`, which could represent continuing a previous square subarray in some context.\n3. The value of `max_with_square_start`, which could represent starting a new square subarray with the current element.\n\nThe purpose of this line is to determine the maximum value that can be achieved by either extending the current square subarray, continuing a previous one, or starting a new one with the current element. This is a common pattern in dynamic programming problems where subarray or substring properties are being optimized.',
    'This line updates the variable `max_with_square_ending_here` by taking the maximum of three possible values:\n1. The sum of the current value of `max_with_square_ending_here` and the current element `nums[i]`. This option represents extending the existing sequence that ends with a square number.\n2. The value of `max_with_square_continue`, which presumably holds the maximum sum of a sequence that continues from the previous elements without breaking the square number condition.\n3. The value of `max_with_square_start`, which represents starting a new sequence with the current element `nums[i]`.\n\nThe purpose of this line is to determine the maximum sum of a sequence ending at the current element `nums[i]` while ensuring that the sequence can be extended or started anew if necessary, all the while maintaining the condition that the sequence includes square numbers.'
]

In [59]:
embeddings = model.encode(explanations)
embeddings

array([[-0.04920113, -0.00019255,  0.02192265, ...,  0.00794916,
         0.00376903,  0.04086377],
       [-0.02688464,  0.04566858, -0.00168553, ...,  0.0374669 ,
        -0.00334383,  0.02675275],
       [-0.06685519,  0.06618014,  0.01415694, ...,  0.03392966,
        -0.00229752,  0.0276531 ]], dtype=float32)

In [60]:
similarities = model.similarity(embeddings, embeddings)
similarities

tensor([[1.0000, 0.8090, 0.8787],
        [0.8090, 1.0000, 0.8531],
        [0.8787, 0.8531, 1.0000]])

In [61]:
def get_similarity_sentences(similarities):
    len_explanations = len(similarities[0]) - 1
    sim_sum = 0
    for sim in similarities[0][1:]:
        sim_sum = sim_sum + sim
        
    return sim_sum/len_explanations

In [62]:
get_similarity_sentences(similarities)

tensor(0.8439)

# Compute the Similarity for the Python Generated Explanations
The explanations are available in the `datasets/python_14B_explanations_5_samples_refined_1.csv` file. There are 5 explanations for each line. I will compute the similarity for pair, starting with 2 explanations and ending up using all the existing explanations. I will add the similarity measures in the `datasets/probabilities_python_final.csv` as new columns: **similarity_2, similarity_3, similarity_4, similarity_5**.

First, I need to parse the explanations and save them per line.

In [43]:
explanations = pd.read_csv('datasets/python_14B_explanations_5_samples_refined_1.csv')

In [37]:
selected_problems = pd.read_csv('datasets/selected_problems.csv')

In [38]:
explanations.head()

,title,sample_number,explanation
0,Sports Rank,1,``` [Line 1]: Explanation: Defines a function ...
1,Sports Rank,2,``` [Line 1]: Explanation: Defines a function ...
2,Sports Rank,3,[Line 1]: Explanation: Define a function named...
3,Sports Rank,4,[Line 1]: Explanation: Define a function named...
4,Sports Rank,5,[Line 1]: Explanation: Define a function named...


In [39]:
titles_selected_problems = list(selected_problems['title'])
titles_selected_problems

['Sports Rank',
 'Solidus Fraction',
 'Breakout',
 'Passport Dates',
 'Weather Forecast',
 'Type Faster',
 'Marathon',
 'Diggy Hole',
 'Hair Dyeing',
 'Date Picker',
 'Contrarianism',
 'Mouse Pursuit',
 'Aggressive Cows',
 'MKTHNUM - K-th Number',
 'HOTELS - Hotels Along the Croatian Coast',
 'LCA - Lowest Common Ancestor',
 'QTREE - Query on a tree',
 'UPDATEIT - Update the array!',
 'A. Way Too Long Words',
 'C. Dijkstra?',
 'G. Game of Chance',
 'E. Organizing a Race',
 'A+B Again?',
 'Party',
 'Zebra-like Numbers',
 'Morse Code',
 'Set of Strangers',
 'Amusing numbers',
 'Time to live',
 'New Distinct Substrings']

In [47]:
explanation_pattern = r'\[Line (\d+)\]: Explanation: (.+)'
def get_explanations(all_explanations, debug = False):
    explanations_per_line = {}
    for sample_explanations in all_explanations:
        lines = sample_explanations.strip().replace('``` ','').split('\n')
        for line in lines:
            try:
                line_explanation = re.search(explanation_pattern, line)
                explanation = line_explanation.group(2)
                line_number = line_explanation.group(1)
                if line_number in explanations_per_line:
                    explanations_per_line[line_number].append(explanation)
                else:
                    explanations_per_line[line_number] = [explanation]
            except:
                if debug:
                    print(line)
    return explanations_per_line
        
get_explanations(list(explanations[explanations['title'] == 'Sports Rank']['explanation']))['1']

['Defines a function named `read_input` to read input data.',
 'Defines a function named `read_input` which is responsible for reading the input data.',
 'Define a function named `read_input` that will be used to read the input data.',
 'Define a function named `read_input` to read input data.',
 'Define a function named `read_input` to handle reading input data.']

In [52]:
problem_explanations_per_line = {}
for problem_title in titles_selected_problems:
    problem_explanations = list(explanations[explanations['title'] == problem_title]['explanation'])
    problem_explanations_per_line[problem_title] = get_explanations(problem_explanations)
    
problem_explanations_per_line['G. Game of Chance']['1']

['Define a function `find_power_of_two` that takes an integer `n` as input.',
 'Define a function `find_power_of_two` that takes an integer `n` as input.',
 'Define a function named `find_power_of_two` that takes an integer `n` as input.',
 'Define a function `find_power_of_two` that takes an integer `n` as input and returns the highest power of two less than or equal to `n`.',
 'Define a function named `find_power_of_two` that takes an integer `n` as input.']

In [76]:
def get_similarity_measure(k, all_problem_explanations_per_line):
    if k > 5 or k < 2:
        print("Invalid `k` parameter value")
    k_explanations = all_problem_explanations_per_line[:k]
    embeddings = model.encode(k_explanations)
    similarities = model.similarity(embeddings, embeddings)
    return get_similarity_sentences(similarities).item()

get_similarity_measure(5, problem_explanations_per_line['G. Game of Chance']['9'])

0.665805459022522

## Storing the simlarity in `datasets/probabilities_python_final.csv`

In [138]:
python_probabilities = pd.read_csv('datasets/probabilities_python_final.csv')

In [139]:
python_probabilities['similarity_2'] = ''
python_probabilities['similarity_3'] = ''
python_probabilities['similarity_4'] = ''
python_probabilities['similarity_5'] = ''

In [140]:
python_probabilities = python_probabilities[python_probabilities['line_number'] != 0]

In [141]:
python_solutions = pd.read_csv('datasets/selected_problems_python_14B_no_comments_final.csv')

In [142]:
python_solutions.head()

,title,description,difficulty,source,solution,solution_no_comments
0,Sports Rank,Sports leagues rank their participants (teams ...,Medium,Kattis,def read_input():\n import sys\n input =...,def read_input():\n import sys\n input =...
1,Solidus Fraction,A fraction written with a solidus (/) can be w...,Medium,Kattis,"def gcd(a, b):\n while b != 0:\n a, ...","def gcd(a, b):\n while b != 0:\n a, ..."
2,Breakout,A variant of the popular video game breakout u...,Medium,Kattis,"def simulate_brick_wall(b, w, h, balls):\n ...","def simulate_brick_wall(b, w, h, balls):\n ..."
3,Passport Dates,Konráð has managed to obtain a big pile of Ice...,Easy,Kattis,def convert_date_format(input_date):\n # Sp...,def convert_date_format(input_date):\n part...
4,Weather Forecast,One of the most important things you can do in...,Hard,Kattis,def parse_input():\n import sys\n input ...,def parse_input():\n import sys\n input ...


### Deleting the useless rows (empty, comments, etc.) from the probability dataset

In [ ]:
def delete_useless_lines(df, probs, comment):
    for index, row in probs.iterrows():
        line = row['line_text'].strip()
        matched_rows = df.loc[df['solution'].str.contains(line, regex=False) & (df['title'] == row['title'])]
        print(matched_rows)
        if matched_rows.empty or line == '' or line.startswith('#'):
            probs.drop(index, inplace=True)
            # print(f"Line: {line}")
                
delete_useless_lines(python_solutions, python_probabilities, '#')

In [144]:
python_probabilities['line_number'] = python_probabilities.groupby('title').cumcount()+1

In [145]:
python_probabilities[python_probabilities['title'] == 'Solidus Fraction']

,title,line_number,line_text,token_prob,similarity_2,similarity_3,similarity_4,similarity_5
56,Solidus Fraction,1,"def gcd(a, b):\n",0.924086,,,,
57,Solidus Fraction,2,while b != 0:\n,0.895459,,,,
58,Solidus Fraction,3,"a, b = b, a % b\n",0.999934,,,,
59,Solidus Fraction,4,return a\n\n,0.983242,,,,
60,Solidus Fraction,5,def main():\n,0.692792,,,,
62,Solidus Fraction,6,numerator_str = input().strip()\n,0.831416,,,,
63,Solidus Fraction,7,vinculum = input().strip()\n,0.931523,,,,
64,Solidus Fraction,8,denominator_str = input().strip()\n\n,0.774990,,,,
66,Solidus Fraction,9,numerator = int(numerator_str)\n,0.997817,,,,
67,Solidus Fraction,10,denominator = int(denominator_str)\n\n,0.999951,,,,


### Add Similarities for 2 to 5 Explanations

In [146]:
for index, row in python_probabilities.iterrows():
    sim_2 = get_similarity_measure(2, problem_explanations_per_line[row['title']][str(row['line_number'])])
    sim_3 = get_similarity_measure(3, problem_explanations_per_line[row['title']][str(row['line_number'])])
    sim_4 = get_similarity_measure(4, problem_explanations_per_line[row['title']][str(row['line_number'])])
    sim_5 = get_similarity_measure(5, problem_explanations_per_line[row['title']][str(row['line_number'])])
    
    python_probabilities.loc[index, "similarity_2"] = sim_2
    python_probabilities.loc[index, "similarity_3"] = sim_3
    python_probabilities.loc[index, "similarity_4"] = sim_4
    python_probabilities.loc[index, "similarity_5"] = sim_5

In [147]:
python_probabilities.head()

,title,line_number,line_text,token_prob,similarity_2,similarity_3,similarity_4,similarity_5
1,Sports Rank,1,def read_input():\n,0.667149,0.971578,0.945252,0.946623,0.945271
2,Sports Rank,2,import sys\n,0.972428,1.0,0.887545,0.922889,0.940561
3,Sports Rank,3,input = sys.stdin.read\n,0.992203,0.87259,0.833801,0.862679,0.879646
4,Sports Rank,4,data = input().split()\n,0.992855,0.899708,0.814728,0.864639,0.822439
6,Sports Rank,5,index = 0\n,0.939398,0.884853,0.914327,0.931024,0.936494


In [148]:
python_probabilities.to_csv('datasets/all_probabilities.csv', index=False)